# 3. Longest Substring Without Repeating Characters
**Difficulty:** 🟡 Medium · **Topic:** String · **LeetCode:** https://leetcode.com/problems/longest-substring-without-repeating-characters/

## 💡 Concepts

**Core concept(s):** A **sliding window** plus a **hash set/map** to track what's currently inside.

**Why it applies here:** We want the longest stretch with no repeats. Keep a window that always holds only unique characters. When a repeat tries to enter, shrink the window from the left until the repeat is gone. Each character enters and leaves once — one pass.

**Key intuition:** Grow the window on the right; the moment a character repeats, slide the left edge past its previous copy.

---

### 📚 What is a Sliding Window?
A **window** is a range `[left, right]` over the string that you grow on the right and shrink on the left, keeping some running summary (a count, a set) as it moves. You never re-scan from scratch.
- **Complexity:** each character enters and leaves the window at most once → **O(n)** total.
- **In Python:** two indices plus a `dict`/`set`/`Counter` describing what's inside.

### 📚 What is a Hash Map / Hash Set?
A **hash map** (Python `dict`) stores **key → value** pairs; a **hash set** (`set`) stores unique keys. Both use a *hash function* to jump straight to a slot instead of scanning.
- **Operations & complexity:** insert / lookup / delete are **O(1) on average**.
- **In Python:** `dict` for counts/mappings, `set` for "have I seen this?". `collections.Counter` counts items for you.

---

**Prerequisite knowledge:**
- Sliding window with two indices.
- A `set` (seen chars) or `dict` (char -> last position).

## 📝 Problem

Return the length of the longest substring with **no repeating characters**.

**Example**
```
"abcabcbb" -> 3   ("abc")
"bbbbb"    -> 1   ("b")
"pwwkew"   -> 3   ("wke")
```

### Approach 1 — Check Every Start (worst)

**Idea:** From each start, extend while characters stay unique (a small set); record the longest.

**Time complexity:** `O(n^2)`.

**Space complexity:** `O(n)` for the set.

In [ ]:
def length_brute(s: str) -> int:
    best = 0
    for i in range(len(s)):                # try every possible starting index
        seen = set()                       # characters used in the current window
        for j in range(i, len(s)):         # extend the window to the right
            if s[j] in seen:               # a repeat means this window can't grow further
                break                      # stop; move on to the next start i
            seen.add(s[j])                 # record the new character
            best = max(best, j - i + 1)    # window length = j - i + 1
    return best

### Approach 2 — Sliding Window with a Set (better)

**Idea:** Keep a window of unique characters. When `s[r]` is already inside, drop characters from the left until it's gone, then add `s[r]`.

**Time complexity:** `O(n)`.

**Space complexity:** `O(n)`.

In [ ]:
def length_window_set(s: str) -> int:
    seen = set()                           # characters currently inside the window
    left = best = 0                        # left edge of the window; best length so far
    for right in range(len(s)):            # right edge sweeps across the string
        while s[right] in seen:            # new char already inside -> shrink from the left
            seen.remove(s[left])           # drop the leftmost char
            left += 1                      # move the left edge right
        seen.add(s[right])                 # now safe to add the new character
        best = max(best, right - left + 1) # update the best window length
    return best

### Approach 3 — Sliding Window with Last-Seen Map (optimal)

**Idea:** Remember each character's **last position**. On a repeat, jump `left` directly past that position instead of shrinking one step at a time.

**Time complexity:** `O(n)`.

**Space complexity:** `O(n)`.

In [ ]:
def length_window_map(s: str) -> int:
    last = {}                              # character -> the last index we saw it at
    left = best = 0                        # window's left edge; best length so far
    for right, c in enumerate(s):          # right edge sweeps across the string
        # If c was seen and its last position is inside the window, jump left past it.
        if c in last and last[c] >= left:
            left = last[c] + 1             # skip straight past the previous copy
        last[c] = right                    # remember where we saw c this time
        best = max(best, right - left + 1) # update the best window length
    return best

In [ ]:
# Correctness check
tests = [("abcabcbb",3), ("bbbbb",1), ("pwwkew",3), ("",0), ("dvdf",3), ("abba",2)]
for s, exp in tests:
    a, b, c = length_brute(s), length_window_set(s), length_window_map(s)
    print(f"{s!r:>10} -> brute={a}, set={b}, map={c} | expected={exp}")
    assert a == b == c == exp, "mismatch!"
print("\nAll tests passed")

## ⏱️ Empirically Checking the Complexities

Big-O can't be read off a function directly, but it can be **measured**. We time each approach on inputs of growing `n` and read the **doubling ratio** — how much runtime grows when `n` doubles.

| Theoretical | Ratio when `n` → `2n` |
|-------------|-----------------------|
| `O(n)`        | ≈ **2×** |
| `O(n log n)`  | ≈ **2×** (slightly more) |
| `O(n²)`       | ≈ **4×** |
| `O(n³)`       | ≈ **8×** |

Inputs are built to force the **worst case** (no early exit). Sub-millisecond rows are noisy — look at the trend.

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark   # shared: prints ratio table + optional log-log plot

def make_worst_case(n):
    s = ("abc" * (n // 3 + 1))[:n]          # many repeats -> brute keeps restarting
    return (s,)

solutions = {
    "brute  O(n^2)": length_brute,
    "set    O(n)  ": length_window_set,
    "map    O(n)  ": length_window_map,
}
sizes = [1000, 2000, 4000, 8000]

benchmark(solutions, make_worst_case, sizes, plot=True)


## 🧩 Patterns Learned

- **Sliding window for "longest/shortest stretch that satisfies a rule":** grow on the right, shrink on the left, keep a running summary — `O(n)`.
- **Last-seen map to jump:** remembering positions lets the left edge leap instead of crawl.
- **Signal:** "longest/shortest substring with (no repeats / at most k / containing ...)".
- **Related problems:** Longest Repeating Character Replacement, Minimum Window Substring, Find All Anagrams.
- **Common pitfalls:** (1) not checking `last[c] >= left` (stale positions); (2) forgetting to update the best length each step.